This notebook uses the MinerU library to convert the selected hydrology research papers
from PDF format into structured Markdown, JSON, and layout visualizations.

It reads the manually downloaded PDF files from the 'Houston_pdfs/' folder
and creates an output folder for each paper under 'Houston_pdfs/output/{IDPaper}/'.

Each output folder contains:
- Extracted content in Markdown format
- Intermediate JSON files for further analysis
- Layout and span visualizations for debugging

In [ ]:
# Import libraries

import os
from magic_pdf.data.data_reader_writer import FileBasedDataWriter, FileBasedDataReader
from magic_pdf.data.dataset import PymuDocDataset
from magic_pdf.model.doc_analyze_by_custom_model import doc_analyze
from magic_pdf.config.enums import SupportedPdfParseMethod

# Enable fallback for MPS in case of GPU issues (Mac compatibility)
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [ ]:
# ---------- Configuration ----------

input_dir = "./Houston_pdfs"
output_dir = "./Houston_pdfs/output"
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# ---------- Processing Loop ----------

for pdf_file in os.listdir(input_dir):
    if not pdf_file.endswith(".pdf"):
        continue

    paper_id = os.path.splitext(pdf_file)[0]
    pdf_path = os.path.join(input_dir, pdf_file)
    paper_output_dir = os.path.join(output_dir, paper_id)

    # Skip already processed files
    if os.path.exists(os.path.join(paper_output_dir, "content.md")):
        print(f"Skipping {pdf_file}, already processed.")
        continue

    os.makedirs(paper_output_dir, exist_ok=True)
    image_dir = os.path.join(paper_output_dir, "images")
    os.makedirs(image_dir, exist_ok=True)

    image_writer = FileBasedDataWriter(image_dir)
    md_writer = FileBasedDataWriter(paper_output_dir)

    try:
        # Load PDF bytes
        reader = FileBasedDataReader("")
        pdf_bytes = reader.read(pdf_path)

        # Initialize document dataset
        dataset = PymuDocDataset(pdf_bytes)

        # Select parsing strategy: OCR or direct text
        if dataset.classify() == SupportedPdfParseMethod.OCR:
            infer_result = dataset.apply(doc_analyze, ocr=True)
            parsed_result = infer_result.pipe_ocr_mode(image_writer)
        else:
            infer_result = dataset.apply(doc_analyze, ocr=False)
            parsed_result = infer_result.pipe_txt_mode(image_writer)

        # Save visualizations and extracted content
        infer_result.draw_model(os.path.join(paper_output_dir, "model.pdf"))
        parsed_result.draw_layout(os.path.join(paper_output_dir, "layout.pdf"))
        parsed_result.draw_span(os.path.join(paper_output_dir, "spans.pdf"))
        parsed_result.dump_md(md_writer, "content.md", image_dir)
        parsed_result.dump_content_list(md_writer, "content_list.json", image_dir)
        parsed_result.dump_middle_json(md_writer, "middle.json")

        print(f"✓ Processed: {pdf_file}")

    except Exception as e:
        print(f"✗ Error processing {pdf_file}: {e}")